# 02 — VQA Generation

Parses WebUOT-238-Test annotations and generates LLaVA-format VQA training pairs for Cosmos-Reason2-2B SFT.

**Run order:** Cell 1 (install) → Cell 2 (mount + config) → Cell 3 (load dataset) → Cell 4 (smoke test) → Cell 5 (full generation) → Cell 6 (verify)

## Cell 1 — Install

In [ ]:
!pip install -U fiftyone -q

## Cell 2 — Mount Drive & config

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, random
from collections import defaultdict
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# ── Config ───────────────────────────────────────────────────────────────────
SEED         = 42
VAL_RATIO    = 0.15
OUTPUT_TRAIN = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_train.json"
OUTPUT_VAL   = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val.json"
VIDEO_DIR    = "/content/drive/MyDrive/cosmos-cookoff/data/videos"

random.seed(SEED)
os.makedirs(os.path.dirname(OUTPUT_TRAIN), exist_ok=True)
print("Config ready.")

## Cell 3 — Define attributes, helpers, and QA generators

In [ ]:
# ── Attribute definitions ────────────────────────────────────────────────────
BINARY_ATTRIBUTES = [
    "Low Resolution", "Fast Motion", "Scale Variations", "Aspect Ratio Variations",
    "Camera Motion", "Viewpoint Changes", "Partial Occlusion", "Full Occlusion",
    "Out-of-View", "Rotation", "Deformation", "Similar Distractors",
    "Illumination Variations", "Motion Blur", "Partial Target Information",
    "Camouflage",
]

CATEGORICAL_ATTRIBUTES = {
    "Natural or Artificial Object": ["natural", "artificial"],
    "Underwater Visibility":        ["low", "medium", "high"],
    "Watercolor Variations":        ["colorless", "blue", "green", "turbid"],
    "Underwater Scenarios":         ["sea", "river", "lake", "fish tank"],
    "Shooting Perspective":         ["underwater", "above water", "surface"],
    "Size":                         ["small", "medium", "large"],
    "Length":                       ["short", "medium", "long"],
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def get_attr(sample, attr_name):
    """Safely get attribute label from a FiftyOne sample."""
    try:
        val = sample[attr_name]
        return val.label if val is not None else None
    except Exception:
        return None

def bbox_to_quadrant(x, y, w, h):
    """Convert normalized XYWH bbox to human-readable position description."""
    cx = x + w / 2
    cy = y + h / 2
    h_pos = "left" if cx < 0.33 else ("center" if cx < 0.67 else "right")
    v_pos = "top"  if cy < 0.33 else ("middle" if cy < 0.67 else "bottom")
    if h_pos == "center" and v_pos == "middle":
        return "center of the frame"
    elif h_pos == "center":
        return f"{v_pos} of the frame"
    elif v_pos == "middle":
        return f"{h_pos} side of the frame"
    else:
        return f"{v_pos}-{h_pos} of the frame"

def get_representative_bbox(sample):
    """Get bounding box from the middle frame (most representative)."""
    frame_nums = sorted(sample.frames.keys())
    if not frame_nums:
        return None
    mid   = frame_nums[len(frame_nums) // 2]
    frame = sample.frames[mid]
    if frame.gt and frame.gt.detections:
        det = frame.gt.detections[0]
        return det.bounding_box, det.label, det.supercategory
    return None

def get_visibility_stats(sample):
    """Compute average visibility across frames."""
    visibilities = []
    for frame in sample.frames.values():
        if frame.gt and frame.gt.detections:
            for det in frame.gt.detections:
                if det.visibility is not None:
                    visibilities.append(det.visibility)
    if not visibilities:
        return None
    avg = sum(visibilities) / len(visibilities)
    return "visible" if avg > 0.5 else "partially or fully occluded"

def build_hazard_list(sample):
    """Build a human-readable hazard description from binary attributes."""
    hazard_map = {
        "Full Occlusion":           "target fully occluded by obstacles",
        "Out-of-View":              "target moves out of camera view",
        "Motion Blur":              "motion blur reducing visual clarity",
        "Similar Distractors":      "similar-looking objects that could confuse tracking",
        "Camouflage":               "target camouflage making detection difficult",
        "Fast Motion":              "fast target motion requiring rapid response",
        "Partial Occlusion":        "partial occlusion reducing target visibility",
        "Illumination Variations":  "illumination changes affecting detection",
        "Deformation":              "target shape changes complicating tracking",
    }
    hazards = []
    for attr, desc in hazard_map.items():
        val = get_attr(sample, attr)
        if val and "present" in val.lower():
            hazards.append(desc)
    return hazards

# ── QA generators ─────────────────────────────────────────────────────────────
def make_object_id_qa(sample, video_path, idx):
    prompt = sample.language if sample.language else None
    bbox_info = get_representative_bbox(sample)
    if not prompt or not bbox_info:
        return None
    bbox, label, _ = bbox_info
    x, y, w, h     = bbox
    position        = bbox_to_quadrant(x, y, w, h)
    answer = f"The {prompt}. It is located in the {position}."
    return {
        "id": f"sample_{idx:04d}_obj_id",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": f"<video>\nWhat underwater subject is the focus of this video clip? Describe what you observe."},
            {"from": "gpt",   "value": answer},
        ]
    }

def make_scene_assessment_qa(sample, video_path, idx):
    scenario   = get_attr(sample, "Underwater Scenarios")
    visibility = get_attr(sample, "Underwater Visibility")
    perspective = get_attr(sample, "Shooting Perspective")
    if not scenario or not visibility:
        return None
    vis_text = {
        "low":    "low, which presents significant challenges for AUV navigation — the camera range is limited and target detection is difficult",
        "medium": "moderate, allowing for adequate AUV navigation with some limitations in detection range",
        "high":   "high, providing good conditions for AUV navigation and target detection",
    }.get(visibility, visibility)
    answer = (
        f"This footage is from a {scenario} environment"
        + (f", filmed from an {perspective} perspective" if perspective else "")
        + f". Underwater visibility is {vis_text}."
    )
    return {
        "id": f"sample_{idx:04d}_scene",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": "<video>\nAssess the underwater visibility and environmental conditions in this scene. How suitable are these conditions for autonomous underwater vehicle (AUV) navigation?"},
            {"from": "gpt",   "value": answer},
        ]
    }

def make_hazard_detection_qa(sample, video_path, idx):
    hazards = build_hazard_list(sample)
    if not hazards:
        return None
    if len(hazards) == 1:
        answer = f"The primary navigation challenge is: {hazards[0]}. AUV navigation systems must handle this condition carefully."
    else:
        hazard_list = "; ".join(hazards)
        answer = f"Multiple navigation hazards are present in this scene: {hazard_list}. AUV navigation systems must handle these conditions carefully."
    return {
        "id": f"sample_{idx:04d}_hazard",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": "<video>\nWhat navigation hazards or visual challenges are present in this underwater scene that an autonomous underwater vehicle (AUV) would need to handle?"},
            {"from": "gpt",   "value": answer},
        ]
    }

def make_spatial_qa(sample, video_path, idx):
    bbox_info = get_representative_bbox(sample)
    prompt    = sample.language
    if not bbox_info or not prompt:
        return None
    bbox, label, _ = bbox_info
    x, y, w, h     = bbox
    position        = bbox_to_quadrant(x, y, w, h)
    # Estimate motion direction from first vs last frame bbox
    frame_nums = sorted(sample.frames.keys())
    direction  = "indeterminate"
    if len(frame_nums) >= 2:
        first_frame = sample.frames[frame_nums[0]]
        last_frame  = sample.frames[frame_nums[-1]]
        def get_center(frame):
            if frame.gt and frame.gt.detections:
                d = frame.gt.detections[0]
                b = d.bounding_box
                return b[0] + b[2]/2, b[1] + b[3]/2
            return None, None
        fx, fy = get_center(first_frame)
        lx, ly = get_center(last_frame)
        if fx is not None and lx is not None:
            dx = lx - fx
            dy = ly - fy
            if abs(dx) > abs(dy):
                direction = "toward the right" if dx > 0 else "toward the left"
            else:
                direction = "downward" if dy > 0 else "upward"
    answer = f"The {prompt} is located in the {position}. The {label} is moving {direction}."
    return {
        "id": f"sample_{idx:04d}_spatial",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": "<video>\nWhere is the primary subject located in this underwater scene, and what direction is it moving? This information is important for AUV target tracking."},
            {"from": "gpt",   "value": answer},
        ]
    }

def make_attribute_mcq(sample, video_path, attr, idx):
    val = get_attr(sample, attr)
    if val is None:
        return None
    present = "present" in val.lower()
    label_present = "A) present"
    label_absent  = "B) absent"
    answer = label_present if present else label_absent
    question_map = {
        "Low Resolution":           "Is the video resolution low, making it difficult to identify the target clearly?",
        "Fast Motion":              "Does the target exhibit fast motion that challenges tracking?",
        "Scale Variations":         "Does the target undergo significant scale variations throughout the video?",
        "Aspect Ratio Variations":  "Are there significant aspect ratio variations of the target?",
        "Camera Motion":            "Is there significant camera motion in this video?",
        "Viewpoint Changes":        "Are there significant viewpoint changes that challenge tracking?",
        "Partial Occlusion":        "Is the target partially occluded at any point in this video?",
        "Full Occlusion":           "Is the target ever fully occluded in this video?",
        "Out-of-View":              "Does the target move out of the camera view at any point?",
        "Rotation":                 "Does the target exhibit significant rotation?",
        "Deformation":              "Does the target undergo significant shape deformation?",
        "Similar Distractors":      "Are there similar-looking objects that could distract the tracker?",
        "Illumination Variations":  "Are there significant illumination variations throughout this video?",
        "Motion Blur":              "Is there noticeable motion blur in this video?",
        "Partial Target Information": "Is only partial target information available for tracking?",
        "Camouflage":               "Is the target camouflaged or difficult to distinguish from the background?",
    }
    question_text = question_map.get(attr, f"Is the attribute '{attr}' present in this video?")
    return {
        "id": f"sample_{idx:04d}_mcq_{attr.lower().replace(' ', '_')}",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": f"<video>\n{question_text}\n\n{label_present}\n{label_absent}"},
            {"from": "gpt",   "value": answer},
        ]
    }

def make_categorical_mcq(sample, video_path, attr, choices, idx):
    val = get_attr(sample, attr)
    if val is None:
        return None
    letters = ["A", "B", "C", "D", "E"]
    options = "\n".join(f"{letters[i]}) {c}" for i, c in enumerate(choices))
    matched = next((f"{letters[i]}) {c}" for i, c in enumerate(choices) if c.lower() in val.lower()), None)
    if not matched:
        return None
    question_map = {
        "Natural or Artificial Object": "Is the tracked subject a natural or artificial object?",
        "Underwater Visibility":        "How would you describe the underwater visibility?",
        "Watercolor Variations":        "How would you describe the water color/turbidity?",
        "Underwater Scenarios":         "What type of underwater environment is shown?",
        "Shooting Perspective":         "What is the shooting perspective of this video?",
        "Size":                         "How would you describe the size of the tracked subject relative to the frame?",
        "Length":                       "How would you describe the length of the tracked subject?",
    }
    question_text = question_map.get(attr, f"What is the value of '{attr}'?")
    return {
        "id": f"sample_{idx:04d}_cat_mcq_{attr.lower().replace(' ', '_')}",
        "video": video_path,
        "conversations": [
            {"from": "human", "value": f"<video>\n{question_text}\n\n{options}"},
            {"from": "gpt",   "value": matched},
        ]
    }

# ── Main generation function ───────────────────────────────────────────────────
def generate_qa_pairs(dataset, video_dir):
    all_pairs = []
    skipped   = 0
    qa_index  = 0

    for i, sample in enumerate(dataset):
        video_path = sample.filepath
        if not video_path or not os.path.exists(video_path):
            skipped += 1
            continue

        # 1. Object Identification
        qa = make_object_id_qa(sample, video_path, qa_index)
        if qa: all_pairs.append(qa); qa_index += 1

        # 2. Scene Assessment
        qa = make_scene_assessment_qa(sample, video_path, qa_index)
        if qa: all_pairs.append(qa); qa_index += 1

        # 3. Hazard Detection
        qa = make_hazard_detection_qa(sample, video_path, qa_index)
        if qa: all_pairs.append(qa); qa_index += 1

        # 4. Spatial Reasoning
        qa = make_spatial_qa(sample, video_path, qa_index)
        if qa: all_pairs.append(qa); qa_index += 1

        # 5. Binary attribute MCQs
        for attr in BINARY_ATTRIBUTES:
            qa = make_attribute_mcq(sample, video_path, attr, qa_index)
            if qa: all_pairs.append(qa); qa_index += 1

        # 6. Categorical MCQs — Visibility + Scenario + Size
        for j, (attr, choices) in enumerate(CATEGORICAL_ATTRIBUTES.items()):
            if attr in ("Underwater Visibility", "Underwater Scenarios", "Size"):
                qa = make_categorical_mcq(sample, video_path, attr, choices, j)
                if qa: all_pairs.append(qa); qa_index += 1

        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{len(dataset)} samples, {len(all_pairs)} QA pairs so far")

    print(f"\nTotal QA pairs: {len(all_pairs)} | Skipped videos: {skipped}")
    return all_pairs


def print_stats(pairs, label="Dataset"):
    """Print QA type distribution."""
    type_counts = defaultdict(int)
    for p in pairs:
        pid = p["id"]
        if   "_obj_id"   in pid: type_counts["Object Identification"] += 1
        elif "_scene"    in pid: type_counts["Scene Assessment"]      += 1
        elif "_hazard"   in pid: type_counts["Hazard Detection"]      += 1
        elif "_spatial"  in pid: type_counts["Spatial Reasoning"]     += 1
        elif "_cat_mcq_" in pid: type_counts["Categorical MCQ"]       += 1
        elif "_mcq_"     in pid: type_counts["Binary MCQ"]            += 1
    print(f"\n{label} QA type distribution:")
    for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {k:30s}: {v}")


print("All helpers and QA generators defined.")

## Cell 4 — Load WebUOT-238-Test from HuggingFace

In [ ]:
print("Loading FiftyOne dataset...")
dataset = load_from_hub("Voxel51/WebUOT-238-Test", overwrite=True)
print(f"Loaded {len(dataset)} samples")
print(f"Sample filepath: {dataset.first().filepath}")

## Cell 5 — Smoke test on 5 samples

Verify the pipeline works before running on the full dataset.

In [ ]:
test_dataset = dataset.take(5)
test_pairs   = generate_qa_pairs(test_dataset, VIDEO_DIR)
print_stats(test_pairs, "Smoke test")

print("\n--- Sample open-ended QA ---")
for p in test_pairs[:2]:
    print(f"\nQ: {p['conversations'][0]['value']}")
    print(f"A: {p['conversations'][1]['value']}")

## Cell 6 — Generate full dataset & save to Drive

In [ ]:
print("Generating QA pairs for all 238 samples...")
all_pairs = generate_qa_pairs(dataset, VIDEO_DIR)

# Shuffle and split
random.shuffle(all_pairs)
split_idx = int(len(all_pairs) * (1 - VAL_RATIO))
train = all_pairs[:split_idx]
val   = all_pairs[split_idx:]

# Save to Drive
with open(OUTPUT_TRAIN, "w") as f:
    json.dump(train, f, indent=2)
with open(OUTPUT_VAL, "w") as f:
    json.dump(val, f, indent=2)

print(f"Saved {len(train)} train pairs → {OUTPUT_TRAIN}")
print(f"Saved {len(val)} val pairs   → {OUTPUT_VAL}")

print_stats(train, "Train")
print_stats(val,   "Val")

## Cell 7 — Verify saved files

In [ ]:
with open(OUTPUT_TRAIN) as f:
    train_check = json.load(f)
with open(OUTPUT_VAL) as f:
    val_check = json.load(f)

print(f"Train: {len(train_check)} pairs  ✓")
print(f"Val:   {len(val_check)} pairs  ✓")
print(f"\nSample entry:")
print(json.dumps(train_check[0], indent=2))